<a href="https://colab.research.google.com/github/anshuman-dataverse/Gen_ai/blob/main/M3_Lab1_Prompting_Strategies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!-- applied-genai-header -->
<div style="background: linear-gradient(135deg, #1e4b8f 0%, #2d6cb8 100%); color: white; padding: 24px; border-radius: 12px; font-family: 'Segoe UI', sans-serif; margin-bottom: 16px;">
  <div style="font-size: 12px; opacity: 0.85; letter-spacing: 1.5px; text-transform: uppercase;">Applied Generative AI · IE 5373</div>
  <h1 style="margin: 8px 0 4px 0; font-size: 28px; font-weight: 600;">M3 Lab 1 — Prompting Strategies</h1>
  <div style="font-size: 14px; opacity: 0.9;">Zero-shot, few-shot, chain-of-thought, role/task/format scaffolds</div>
  <div style="margin-top: 12px; font-size: 12px; opacity: 0.8;">Prof. Mohammad Dehghani · Northeastern University</div>
</div>

> **📌 Note on models.** This lab references specific LLM versions (e.g. `gpt-5`, `gpt-5-mini`).
> Models update quickly — you are welcome (and encouraged) to swap in any newer OpenAI / Anthropic / Google model you have access to.
> The default model is set in one place: `DEFAULT_CHAT_MODEL` inside `utils.py`. Change it there and every cell follows.


In [ ]:
# === Shared lab setup: utils.py + API key + sticky lab pill ===
# Downloads the shared utilities (pretty_print, model constants, key loader,
# lab_pill) from the AppliedGenAI repo so every notebook stays small and
# consistent. The API key is read from a Colab secret named OPENAI_API_KEY
# (set it once under Colab → 🔑 → "Notebook access" — same name in every lab).
import os
if not os.path.exists("utils.py"):
    !wget -q https://raw.githubusercontent.com/mdehghani86/AppliedGenAI/main/utils.py -O utils.py

from utils import (
    pretty_print,
    DEFAULT_CHAT_MODEL,   # e.g. "gpt-5"  — main reasoning model
    DEFAULT_MINI_MODEL,   # e.g. "gpt-5-mini"  — cheaper / faster default
    DEFAULT_EMBED_MODEL,  # e.g. "text-embedding-3-small"
    get_openai_key,
    lab_pill,
)

lab_pill('M3 Lab 1 — Prompting Strategies')        # sticky banner so you always see which lab you're in
get_openai_key(verify=True)    # loads the key + pings OpenAI to confirm it works


<a href="https://colab.research.google.com/github/mdehghani86/AppliedGenAI/blob/main/M3_Lab1_Prompting_Strategies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!-- Intro Section -->
<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); color: white; padding: 30px; border-radius: 12px; text-align: center; box-shadow: 0 4px 12px rgba(0,0,0,0.1);">
    <h1 style="margin-bottom: 10px; font-size: 32px;">Introduction to Prompting Strategies</h1>
    <p style="font-size: 18px; margin: 0;">Instructor: <strong>Dr. Dehghani</strong></p>
</div>

<!-- Spacer -->
<div style="height: 30px;"></div>

<!-- Why It Matters Section -->
<div style="background: #ffffff; padding: 25px; border-radius: 10px; border-left: 6px solid #0055d4; box-shadow: 0 4px 8px rgba(0,0,0,0.05);">
    <h2 style="margin-top: 0; color: #001a70;">Why Prompting Strategies Matter</h2>
    <p style="font-size: 16px; line-height: 1.6;">
        Imagine you’re working with a junior engineer. You say:  
        <em>“Optimize the system.”</em><br>
        They’ll probably ask: <em>“Which system? Optimize for cost, speed, or energy? Any constraints?”</em> 🧐
    </p>
    <p style="font-size: 16px; line-height: 1.6;">
        Now try this instead:  
        <em>“Analyze the HVAC system and minimize energy consumption while keeping temperatures between 22-24°C. Provide a cost breakdown.”</em>  
    </p>
    <p style="font-size: 16px; line-height: 1.6;">
        That’s not just a prompt—it’s a <strong>clear strategy</strong> with defined objectives and boundaries.
        And that’s exactly what AI models need to perform at their best.
    </p>
</div>

<!-- Tip Section -->
<div style="background: #f5faff; padding: 20px; border-radius: 8px; border-left: 5px solid #0055d4; margin-top: 30px;">
    <h3 style="margin-top: 0; color: #0055d4;">💡 Pro Tip</h3>
    <p style="margin: 0; font-size: 16px; line-height: 1.6;">
        AI models appreciate well-structured instructions just like engineers appreciate complete design specs.
        Be specific, set clear goals, and watch the results improve!
    </p>
</div>

<!-- Upcoming Topics -->
<div style="margin-top: 40px; text-align: center;">
    <h3 style="color: #001a70;">What’s Ahead</h3>
    <ul style="list-style: none; padding: 0; font-size: 16px; line-height: 1.8;">
        <li>📚 Basic Prompting Types</li>
        <li>🧩 Advanced Strategies</li>
        <li>📊 Application-Specific Techniques</li>
    </ul>
    <p style="font-size: 16px; color: #333;">Let’s engineer some powerful AI conversations! 🛠️</p>
</div>


<!-- Section Header -->
<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); color: white; padding: 25px; border-radius: 12px; text-align: center; box-shadow: 0 4px 12px rgba(0,0,0,0.1);">
    <h1 style="margin-bottom: 10px; font-size: 30px;">📚 Basic Prompting Types</h1>
</div>

<!-- Spacer -->
<div style="height: 25px;"></div>

<!-- Zero-Shot Prompting -->
<div style="background: #ffffff; padding: 20px; border-radius: 10px; border-left: 6px solid #0055d4; margin-bottom: 20px;">
    <h3 style="margin-top: 0; color: #001a70;">1️⃣ Zero-Shot Prompting</h3>
    <p style="font-size: 16px; line-height: 1.6;">
        Provide only the task without any examples.  
        <strong>Use When:</strong> The task is simple and well-known by the model.  
        <em>Example:</em> “Translate 'Hello' to French.”
    </p>
</div>


In [3]:
# ==========================
# 📌 Set Up LLM and OpenAI API
# ==========================
# Import required libraries
from google.colab import userdata
import openai
import os

# Load the OpenAI API key securely from Colab secrets
api_key = userdata.get('OPENAI_API_KEY')

# Check that the API key was found
if api_key is None:
    raise ValueError("❌ API Key not found. Please store your OpenAI API key using Colab secrets.")

# Set API key as environment variable for OpenAI
os.environ["OPENAI_API_KEY"] = api_key

# Initialize OpenAI client
client = openai.OpenAI(api_key=api_key)

print("✅ OpenAI API Key successfully loaded and environment is ready!")

# ==========================
# 📌 Set LLM Model to GPT-3.5
# ==========================
# Define which LLM model to use
model_name = DEFAULT_MINI_MODEL

print(f"✅ LLM model set to: {model_name}")


✅ OpenAI API Key successfully loaded and environment is ready!
✅ LLM model set to: gpt-4o-mini


In [4]:
from openai import OpenAI
client = OpenAI()

# ==========================
# 📌 Zero-Shot Test: Hidden Formula Sequence
# ==========================

hard_sequence_prompt_zero = (
    "The sequence is: 3, 12, 27, 48, 75, ___. What’s next?"
)

response_zero_hard = client.chat.completions.create(
    model=DEFAULT_MINI_MODEL,
    messages=[{"role": "user", "content": hard_sequence_prompt_zero}],
    temperature=0
)

print("🔹 LLM Response (Zero-Shot - Hard Sequence):\n")
pretty_print(response_zero_hard.choices[0].message.content.strip(), title="🤖 Model Response")


🔹 LLM Response (Zero-Shot - Hard Sequence):




<!-- One-Shot Prompting -->
<div style="background: #ffffff; padding: 20px; border-radius: 10px; border-left: 6px solid #0055d4; margin-bottom: 20px;">
    <h3 style="margin-top: 0; color: #001a70;">2️⃣ One-Shot Prompting</h3>
    <p style="font-size: 16px; line-height: 1.6;">
        Provide one clear example along with the instruction.  
        <strong>Use When:</strong> You want to guide the model’s behavior with a single example.  
        <em>Example:</em> “Translate 'Hello' to French: Bonjour. Now translate 'Goodbye'.”
    </p>
</div>


In [5]:
from openai import OpenAI
client = OpenAI()

# ==========================
# 📌 Zero-Shot vs One-Shot Comparison: Alternating Pattern Sequence (Correct One-Shot)
# ==========================

model_name = DEFAULT_MINI_MODEL

# Zero-Shot Prompt (No Example)
zero_shot_prompt = (
    "The sequence is: 1, 4, 2, 9, 3, 16, 4, ___. What number should replace the blank?"
)

# One-Shot Prompt (One Example + New Question)
one_shot_prompt = (
    "Example:\n"
    "The sequence is: 1, 1, 2, 4, 3, 9, ___. What’s next?\n"
    "Answer: 4.\n\n"
    "Now solve this one:\n"
    "The sequence is: 1, 4, 2, 9, 3, 16, 4, ___. What number should replace the blank?"
)

# Run Zero-Shot
response_zero = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": zero_shot_prompt}],
    temperature=0
)

# Run One-Shot
response_one = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": one_shot_prompt}],
    temperature=0
)

# Display Results
print("🔹 Zero-Shot Response:\n" + "-"*40)
pretty_print(response_zero.choices[0].message.content.strip(), title="🤖 Model Response")

print("\n\n🔹 One-Shot Response:\n" + "-"*40)
pretty_print(response_one.choices[0].message.content.strip(), title="🤖 Model Response")


🔹 Zero-Shot Response:
----------------------------------------




🔹 One-Shot Response:
----------------------------------------



<!-- Few-Shot Prompting -->
<div style="background: #ffffff; padding: 20px; border-radius: 10px; border-left: 6px solid #0055d4; margin-bottom: 20px;">
    <h3 style="margin-top: 0; color: #001a70;">3️⃣ Few-Shot Prompting</h3>
    <p style="font-size: 16px; line-height: 1.6;">
        Provide multiple examples to clearly demonstrate the pattern.  
        <strong>Use When:</strong> The task is complex or requires understanding a specific format.  
        <em>Example:</em>  
        - “Translate 'Hello' to French: Bonjour.”  
        - “Translate 'Goodbye' to French: Au revoir.”  
        - “Translate 'Thank you' to French: Merci.”  
        Now translate 'Good night'.
    </p>
</div>

<!-- Spacer -->
<div style="height: 30px;"></div>

<!-- Closing Tip -->
<div style="background: #f5faff; padding: 20px; border-radius: 8px; border-left: 5px solid #0055d4;">
    <h3 style="margin-top: 0; color: #0055d4;">💡 Quick Reminder</h3>
    <p style="margin: 0; font-size: 16px; line-height: 1.6;">
        The more complex the task, the more examples you should provide. But remember, too many examples can make prompts bulky and inefficient.
    </p>
</div>



In [6]:
from openai import OpenAI
client = OpenAI()

# ==========================
# 📌 Few-Shot Prompting Example: Ultra-Hard Pattern (3 Hidden Rules)
# ==========================

model_name = DEFAULT_CHAT_MODEL  # Best for complex reasoning

# Few-Shot Prompt with 2 Examples
few_shot_prompt = (
    "Example 1:\n"
    "The sequence is: 1, 1, 2, 4, 3, 9, ___. What’s next?\n"
    "Answer: 4.\n\n"
    "Example 2:\n"
    "The sequence is: 1, 1, 2, 4, 4, 9, 7, 16, ___. What’s next?\n"
    "Answer: 11.\n\n"
    "Now try this one:\n"
    "The sequence is: 1, 1, 2, 4, 4, 9, 7, 16, 11, ___, 16, 36. What number should replace the blank?"
)

# Run Few-Shot Prompt
response_few = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": few_shot_prompt}],
    temperature=0
)

# Display Result
print("🔹 Few-Shot Prompting (Two Examples Provided):")
print("-" * 40)
pretty_print(response_few.choices[0].message.content.strip(), title="🤖 Model Response")


🔹 Few-Shot Prompting (Two Examples Provided):
----------------------------------------


## 🧠 Advanced Prompting Techniques  

Moving beyond basic prompting methods like zero-shot and few-shot, advanced strategies help enhance the reasoning and adaptability of large language models (LLMs). These techniques guide the model's thought process to handle complex tasks more effectively.

---

### 🔗 Chain-of-Thought (CoT) Prompting  

Chain-of-Thought prompting encourages models to **explain their intermediate reasoning steps**, leading to more transparent and accurate conclusions. By structuring prompts to include logical steps, CoT improves the model’s ability to solve complex reasoning tasks.

**Why is CoT Important?**  
- ✔️ Improves performance on multi-step reasoning tasks.  
- ✔️ Helps produce logically structured and coherent responses.  
- ✔️ Breaks down complex problems into manageable steps.

📖 **Reference:** [Chain-of-Thought Prompting Elicits Reasoning in Large Language Models](https://arxiv.org/abs/2201.11903)

---

*Next, explore practical examples of Chain-of-Thought prompting.*


In [7]:
from openai import OpenAI
client = OpenAI()

# ==========================
# 📌 Chain-of-Thought Demonstration: Make 110 with Five 5's
# ==========================

model_name = DEFAULT_CHAT_MODEL

# Zero-Shot Prompt (No Reasoning Encouraged)
zero_shot_prompt = (
    "Use exactly five 5’s and only four operations (+, -, *, /) and parentheses to make 110."
)

# Chain-of-Thought Prompt (Encourages Step-by-Step Reasoning)
cot_prompt = (
    "Let's solve this step by step.\n"
    "We need to use exactly five 5’s and only four operations (+, -, *, /) and parentheses to make 110.\n"
    "Step 1: Think about how we can combine the 5's to form larger numbers (e.g., 55).\n"
    "Step 2: Try to combine them logically to reach 110.\n"
    "Now, provide the final equation and the answer."
)

# Run Zero-Shot
response_zero = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": zero_shot_prompt}],
    temperature=0
)

# Run Chain-of-Thought
response_cot = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": cot_prompt}],
    temperature=0
)

# Display Results
print("🔹 Zero-Shot Response (No Reasoning Encouraged):\n" + "-" * 50)
pretty_print(response_zero.choices[0].message.content.strip(), title="🤖 Model Response")

print("\n🔹 Chain-of-Thought Response (Reasoning Encouraged):\n" + "-" * 50)
pretty_print(response_cot.choices[0].message.content.strip(), title="🤖 Model Response")


🔹 Zero-Shot Response (No Reasoning Encouraged):
--------------------------------------------------



🔹 Chain-of-Thought Response (Reasoning Encouraged):
--------------------------------------------------


# ✋ Hands-On Experiment: Observations  

📌 **Instructions:**  
- Run your experiments by changing the model type (e.g., `gpt-3.5-turbo`, `gpt-4-turbo`, `gpt-o3`), temperature, and prompt style.  
- You can **either attach a screenshot/image of your results** or **write a brief summary of your observations (max half a page)**.

---

- **Model Used:**  
  - gpt-4o-mini for the initial run, then gpt-4o for comparison.

- **Temperature Setting:**  
  -I tried both 0.0 and 0.5 to see how randomness changed the outputs

- **Zero-Shot Result:**  
  - No. With just the instruction "use five 5's and four operations to make 110", the model jumped to an answer without working it out and produced an equation that did not actually evaluate to 110. It seemed to guess at a plausible looking expression rather than verify it._

- **Chain-of-Thought Result:**  
  - Yes. When I explicitly broke the problem into steps (first combine 5's to form larger numbers like 55, then find a combination that reaches 110), the model arrived at 55 + 55 + 5 - 5 = 110, which checks out. Forcing intermediate reasoning made the difference.

- **Key Takeaways (Max Half Page or Screenshot):**  
  - The biggest learning was that simply adding the phrase "let's solve this step by step" changes the model's behavior more than I expected. With zero-shot, the model behaves like someone answering quickly without thinking. With CoT, it slows down and self-checks.
  - Temperature also mattered. At temperature 0 the answers were consistent but sometimes consistently wrong. At 0.5 the model occasionally found a correct answer through a different path, but other attempts were worse. Higher temperature is useful for exploration, not for precision tasks.
  - The smaller model (gpt-4o-mini) also failed zero-shot more often than the larger one, suggesting CoT becomes more critical the smaller the model is._

---

✍️ *Try at least two models and different temperatures. Compare the results and reflect on how prompting strategies influence performance!*


## 🔁 Self-Consistency Prompting

While Chain-of-Thought (CoT) improves reasoning by encouraging step-by-step thinking, it may still produce **inconsistent or incorrect** answers, especially in complex scenarios.  
**Self-Consistency Prompting** enhances CoT by asking the model to **generate multiple reasoning paths** and then select the most common or consistent final answer.

### Why is Self-Consistency Useful?

- ✅ Reduces random reasoning errors.
- ✅ Boosts reliability on ambiguous or multi-path problems.
- ✅ Often improves performance on mathematical, logical, and symbolic tasks.

📖 **Reference**: [Self-Consistency Improves Chain of Thought Reasoning in Language Models](https://arxiv.org/abs/2203.11171)

---

*Next, we’ll see how Self-Consistency works in action using a complex reasoning example.*


In [8]:
from openai import OpenAI
client = OpenAI()

# ==========================
# 📌 Comparing Chain-of-Thought vs. Self-Consistency Prompting
# ==========================

model_name = DEFAULT_CHAT_MODEL  # Using GPT-4 for better reasoning

# Define the problem prompt
problem_prompt = (
    "If a train travels at 60 miles per hour and leaves at 2 PM, and another train leaves "
    "the same station at 3 PM traveling at 90 miles per hour, when will the second train catch up to the first?"
)

# Chain-of-Thought Prompt (Standard)
cot_prompt = (
    "Let's solve this step by step.\n"
    + problem_prompt
)

# Self-Consistency Prompt: Ask the model to produce multiple reasoning paths
def run_self_consistency(prompt, num_attempts=5):
    answers = []
    for _ in range(num_attempts):
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7  # Add randomness to explore different reasoning paths
        )
        answer = response.choices[0].message.content.strip()
        answers.append(answer)
    return answers

# Run Chain-of-Thought (Single Attempt)
response_cot = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": cot_prompt}],
    temperature=0
)
cot_answer = response_cot.choices[0].message.content.strip()

# Run Self-Consistency (Multiple Attempts)
sc_answers = run_self_consistency(cot_prompt, num_attempts=5)

# Simple Majority Vote to Find Most Consistent Answer
from collections import Counter
most_common_answer = Counter(sc_answers).most_common(1)[0]

# Display Results
print("🔹 Chain-of-Thought Response (Single Attempt):\n" + "-" * 50)
print(cot_answer)

print("\n🔹 Self-Consistency Responses (Multiple Attempts):\n" + "-" * 50)
for idx, ans in enumerate(sc_answers, 1):
    print(f"Attempt {idx}: {ans}")

print("\n🔹 Final Self-Consistency Selected Answer:\n" + "-" * 50)
print(f"Most Common Answer: {most_common_answer[0]}\nAppeared {most_common_answer[1]} times.")


🔹 Chain-of-Thought Response (Single Attempt):
--------------------------------------------------
To solve this problem, we need to determine when the second train will catch up to the first train. Let's break it down step by step:

1. **Determine the head start of the first train:**
   - The first train leaves at 2 PM traveling at 60 miles per hour.
   - By the time the second train leaves at 3 PM, the first train has been traveling for 1 hour.
   - In that 1 hour, the first train travels:
     \[
     \text{Distance} = \text{Speed} \times \text{Time} = 60 \, \text{miles per hour} \times 1 \, \text{hour} = 60 \, \text{miles}
     \]
   - So, the first train has a 60-mile head start.

2. **Determine the relative speed of the second train compared to the first train:**
   - The second train travels at 90 miles per hour.
   - The first train travels at 60 miles per hour.
   - The relative speed of the second train with respect to the first train is:
     \[
     \text{Relative Speed} = 90

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); color: white; padding: 25px; border-radius: 12px; text-align: center;">
    <h1 style="margin-bottom: 10px;">📚 Exploring More Advanced Prompting Strategies</h1>
</div>

<div style="background: #ffffff; padding: 20px; border-radius: 10px; border-left: 6px solid #0055d4; margin-top: 20px;">
    <ul style="font-size: 16px; line-height: 1.8;">
        <li><strong>🧩 Tree-of-Thought (ToT) Prompting:</strong> Explores multiple reasoning paths like a decision tree, helping the model evaluate and compare various solutions before choosing the best one.</li>
        <li><strong>🤖 ReAct (Reasoning and Acting) Prompting:</strong> Combines reasoning steps with actions, including API calls or external tool usage. Ideal for interactive agents and dynamic decision-making tasks.</li>
        <li><strong>🔄 Reflexion Prompting:</strong> Encourages the model to critique its own responses and iteratively improve them, simulating self-correction and learning.</li>
    </ul>
</div>

<div style="margin-top: 40px; text-align: center;">
    <h2 style="color: #001a70;">✋ Hands-On Task: Compare Prompting Strategies</h2>
</div>

<div style="background: #f5faff; padding: 20px; border-radius: 8px; border-left: 5px solid #0055d4;">
    <p style="font-size: 16px;">
        📌 <strong>Task Instructions:</strong><br>
        - Experiment with <strong>Self-Consistency</strong>, <strong>Tree-of-Thought</strong>, and <strong>ReAct</strong> prompting methods.<br>
        - Try to solve the following problem using each method and compare the results.
    </p>
</div>

<div style="background: #ffffff; padding: 20px; border-radius: 10px; border-left: 6px solid #0055d4; margin-top: 20px;">
    <h3>🧠 <strong>Challenge Problem:</strong></h3>
    <p style="font-size: 16px;">A farmer has chickens and rabbits in a cage. There are 35 heads and 94 legs. How many chickens and rabbits are there?</p>
</div>

<div style="margin-top: 40px;">
    <ul style="font-size: 16px; line-height: 1.8;">
        <li>Try different models (e.g., <code>gpt-3.5-turbo</code>, <code>gpt-4-turbo</code>, <code>gpt-o3</code>).</li>
        <li>Experiment with different temperatures (e.g., <code>0.0</code>, <code>0.5</code>, <code>0.7</code>).</li>
        <li>Use both direct prompts and advanced strategies like CoT, Self-Consistency, or ReAct.</li>
    </ul>
</div>

<div style="margin-top: 40px; text-align: center;">
    <h2 style="color: #001a70;">📖 Observations</h2>
</div>

<div style="background: #ffffff; padding: 20px; border-radius: 10px; border-left: 6px solid #0055d4;">
    <ul style="font-size: 16px; line-height: 1.8;">
        <li><strong>Model and Strategy Used:</strong><br> I tested three strategies on gpt-4o-mini and gpt-4o: a direct zero shot prompt, Chain of Thought with step by step reasoning, and Self-Consistency with 5 attempts at temperature 0.7.</li>
        <li><strong>Was the Correct Answer Found?</strong><br> Yes. The correct answer is 23 chickens and 12 rabbits (23 × 2 + 12 × 4 = 46 + 48 = 94 legs, and 23 + 12 = 35 heads). All three strategies eventually reached this, but with different levels of reliability.</li>
        <li><strong>Key Takeaways (Max Half Page or Screenshot):</strong><br> The direct zero shot prompt worked on gpt-4o but failed on gpt-4o-mini, which made an arithmetic slip when subtracting and reported 22 chickens and 13 rabbits before correcting itself when I asked it to verify. Chain of Thought, where I asked the model to set up equations (c + r = 35 and 2c + 4r = 94) and solve them before answering, was the most reliable single shot method and worked consistently across both models. Self Consistency at temperature 0.7 was overkill for this problem since the math is deterministic. All 5 attempts on gpt-4o reached the same answer, so the majority vote did not add value. Where Self-Consistency would actually help is on ambiguous reasoning problems where the model can take multiple valid paths and some are wrong. For pure math like this, CoT alone is enough and Self Consistency just wastes API calls. The general lesson is to match the strategy to the problem. CoT for math, Self Consistency for ambiguity, ReAct for tool use. </li>
    </ul>
</div>

<div style="margin-top: 20px; text-align: center;">
    ✍️ <em>Hint: Try breaking down the problem into equations or ask the model to explain its steps before giving the final answer. Notice which strategies lead to faster and more accurate results!</em>
</div>


In [ ]:
# ==========================
# ✋ Hands-On Code: Try Different Prompting Strategies and Models
# ==========================

# 📝 Instructions:
# - Change 'model_name' to try different models (e.g., DEFAULT_MINI_MODEL, DEFAULT_CHAT_MODEL, "gpt-o3").
# - Adjust 'temperature' to test how creativity affects reasoning.
# - Try Self-Consistency by sampling multiple outputs and comparing answers.
# - Optionally, explore Tree-of-Thought and ReAct patterns by modifying prompts.
# ✅ Your Experiment Starts Here 👇


<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); color: white; padding: 25px; border-radius: 12px; text-align: center;">
    <h1 style="margin-bottom: 10px;">📌 Conclusion</h1>
</div>

<div style="background: #ffffff; padding: 20px; border-radius: 10px; border-left: 6px solid #0055d4; margin-top: 20px;">
    <p style="font-size: 16px; line-height: 1.8;">
        In this hands-on exploration, different advanced prompting strategies were tested to solve reasoning-based challenges.
        Through experimenting with <strong>Chain-of-Thought (CoT)</strong>, <strong>Self-Consistency</strong>, and other methods,
        the following key insights were observed:
    </p>
    <ul style="font-size: 16px; line-height: 1.8;">
        <li>Advanced prompting techniques significantly improve model performance, especially on complex, multi-step problems.</li>
        <li>Changing the <strong>model type</strong> and <strong>temperature</strong> can drastically affect reasoning quality and creativity.</li>
        <li>Some strategies, like <strong>Self-Consistency</strong>, help reduce random errors by exploring multiple reasoning paths.</li>
        <li>For ambiguous or challenging problems, combining strategies (e.g., CoT + Self-Consistency) often leads to the most reliable results.</li>
    </ul>
</div>

<div style="background: #f5faff; padding: 20px; border-radius: 8px; border-left: 5px solid #0055d4; margin-top: 20px;">
    <p style="font-size: 16px; font-style: italic;">
        📖 <em>Remember: Prompt engineering is both an art and a science. The more you experiment, the better you understand how to guide LLMs effectively!</em>
    </p>
</div>

<div style="margin-top: 40px; text-align: center;">
    <h3 style="color: #001a70;">✍️ Final Reflection</h3>
</div>

<div style="background: #ffffff; padding: 20px; border-radius: 10px; border-left: 6px solid #0055d4;">
    <p style="font-size: 16px;">
       What I took away from this lab is that prompting is much more like writing a brief for a colleague than typing a question into a search bar. The same model gave dramatically different answers depending on whether I asked it to think step by step, gave it examples, or just trusted it to figure things out. Choosing the right strategy (and the right combination of model and temperature) is the actual skill, and the size of the model matters less than how clearly you tell it what to do._
    </p>
</div>
